#Notebook - Camada Gold

###Parte 1 - Visão Comercial e Volume de Produtos
Tabela fato de visualização comercial

In [0]:
from pyspark.sql import functions as F

df_pedido_total = spark.table("ecommerce_lakehouse.silver.fat_pedido_total")
df_pedidos      = spark.table("ecommerce_lakehouse.silver.fat_pedidos")
df_itens        = spark.table("ecommerce_lakehouse.silver.fat_pedidos_itens")
df_produtos     = spark.table("ecommerce_lakehouse.silver.dim_produtos")
df_cotacao      = spark.table("ecommerce_lakehouse.silver.dim_cotacao_dolar")
df_traducao     = spark.table("ecommerce_lakehouse.silver.dim_categoria_produtos_traducao")


# dim_categoria_produtos_traducao usa nome_produto_pt como chave
# dim_produtos usa categoria_produto no formato raw (com underscore)
# padronizando antes do join
df_traducao_std = df_traducao.select(
    # Padroniza a chave PT para bater com o raw do dim_produtos
    F.regexp_replace(
        F.lower(F.trim(F.col("nome_produto_pt"))), " ", "_"
    ).alias("categoria_produto_key"),
    F.col("nome_produto_pt").alias("categoria_produto_pt"),
    F.col("nome_produto_en").alias("categoria_produto_en"),
)
#removendo possiveis categorias nulas e fazendo o join produtos com categorias
df_produtos_enriquecido = (
    df_produtos
    .select("id_produto", "categoria_produto")
    .join(df_traducao_std,
          df_produtos["categoria_produto"] == df_traducao_std["categoria_produto_key"],
          how="left")
    .withColumn(
        "categoria_produto_pt",
        F.coalesce(F.col("categoria_produto_pt"), F.lit("Sem Categoria"))
    )
    .select("id_produto", "categoria_produto_pt", "categoria_produto_en")
)

# base analitica com itens + produto + pedido + cambio

df_base = (
    df_itens
    .select("id_pedido", "id_produto", "id_item", "preco_BRL", "preco_frete")
    .join(df_produtos_enriquecido, on="id_produto", how="left")
    .join(
        df_pedidos.select("id_pedido", "status_pedido", "data_compra"),
        on="id_pedido", how="inner"
    )
    .filter(F.col("status_pedido") == "Entregue")
    .join(
        df_cotacao.select("data_cotacao", "valor_cotacao_compra"),
        df_pedidos["data_compra"].cast("date") == df_cotacao["data_cotacao"],
        how="left"
    )
    .withColumn("ano_venda", F.year("data_compra"))
    .withColumn("mes_venda", F.month("data_compra"))
    # Receita = preço + frete, não consideram a soma antes
    .withColumn("receita_item_brl", F.col("preco_BRL") + F.col("preco_frete"))
    .withColumn(
        "receita_item_usd",
        F.round(F.col("receita_item_brl") / F.col("valor_cotacao_compra"), 2)
    )
)

df_gold = (
    df_base
    .groupBy("ano_venda", "mes_venda", "categoria_produto_pt")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.round(F.sum("receita_item_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("receita_item_usd"), 2).alias("receita_total_usd"),
        #ticket medio sendo o total de receita / total de pedidos
        F.round(
            F.sum("receita_item_brl") / F.countDistinct("id_pedido"), 2
        ).alias("ticket_medio_brl"),
    )
    .withColumnRenamed("categoria_produto_pt", "categoria_produto")
    .orderBy("ano_venda", "mes_venda", F.col("receita_total_brl").desc())
    .withColumn("dt_referencia", F.to_date(F.current_timestamp()))
    .withColumn("_gold_load_at", F.current_timestamp())
)

total = df_gold.count()
nulos_cat = df_gold.filter(F.col("categoria_produto").isNull()).count()
nulos_receita = df_gold.filter(F.col("receita_total_brl").isNull()).count()

assert nulos_cat == 0, "Existem nulos em categoria_produto."
assert nulos_receita == 0, "Existem nulos em receita_total_brl."

TABLE_NAME = "ecommerce_lakehouse.gold.fat_vendas_comercial"

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

print(f"Tabela {TABLE_NAME} salva com {total} registros.")




com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

Visualização de Top 5 Itens mais e menos vendidos

In [0]:
df_ranking_base = (
    df_itens
    .select("id_pedido", "id_produto", "id_item")
    .join(
        df_pedidos.select("id_pedido", "status_pedido"),
        on="id_pedido", how="inner"
    )
    .filter(F.col("status_pedido") == "Entregue")
    .join(df_produtos_enriquecido, on="id_produto", how="left")
    .groupBy("id_produto", "categoria_produto_pt")
    .agg(F.count("id_item").alias("quantidade_vendida"))
    .filter(F.col("quantidade_vendida") > 0)
)

df_top5_mais = (
    df_ranking_base
    .orderBy(F.col("quantidade_vendida").desc())
    .limit(5)
    .select(
        F.col("id_produto").alias("produto"),
        F.col("categoria_produto_pt").alias("categoria"),
        F.col("quantidade_vendida"),
    )
)

print("\n TOP 5 Produtos MAIS Vendidos 📈")
display(df_top5_mais)

df_top5_menos = (
    df_ranking_base
    .orderBy(F.col("quantidade_vendida").asc())
    .limit(5)
    .select(
        F.col("id_produto").alias("produto"),
        F.col("categoria_produto_pt").alias("categoria"),
        F.col("quantidade_vendida"),
    )
)

print("\nTOP 5 Produtos MENOS Vendidos 📉")
display(df_top5_menos)



 TOP 5 Produtos MAIS Vendidos 📈


produto,categoria,quantidade_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,520
422879e10f46682990de24d770e7f83d,ferramentas_jardim,484
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,477
389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,390
368c6c730842d78016ad823897a372db,ferramentas_jardim,388



TOP 5 Produtos MENOS Vendidos 📉


produto,categoria,quantidade_vendida
00c744ca2f3b0e76ce227b146095d3f9,brinquedos,1
0044d70d4e53450c0fbb8255446a797b,esporte_lazer,1
00a985c524adbb97a4211e4ce17aabec,telefonia,1
002c6dab60557c48cfd6c2222ef7fd76,brinquedos,1
011bb8a6af5178b33871a83c4f297d32,informatica_acessorios,1


###Parte 2 - Satisfação dos Clientes
Tabela fato de visualização das avaliações

In [0]:
from pyspark.sql import functions as F

df_avaliacoes = spark.table("ecommerce_lakehouse.silver.fat_avaliacoes_pedidos")
df_itens      = spark.table("ecommerce_lakehouse.silver.fat_pedidos_itens")
df_produtos   = spark.table("ecommerce_lakehouse.silver.dim_produtos")
df_vendedores = spark.table("ecommerce_lakehouse.silver.dim_vendedores")
df_traducao   = spark.table("ecommerce_lakehouse.silver.dim_categoria_produtos_traducao")

#Normalizando categoria_produtos_traducao com dim_produtos
df_traducao_std = df_traducao.select(
    F.regexp_replace(
        F.lower(F.trim(F.col("nome_produto_pt"))), " ", "_"
    ).alias("categoria_produto_key"),
    F.col("nome_produto_pt").alias("categoria_produto_pt"),
)

df_produtos_cat = (
    df_produtos
    .select("id_produto", "categoria_produto")
    .join(
        df_traducao_std,
        df_produtos["categoria_produto"] == df_traducao_std["categoria_produto_key"],
        how="left"
    )
    .withColumn(
        "categoria_produto_pt",
        F.coalesce(F.col("categoria_produto_pt"), F.lit("Sem Categoria"))
    )
    .select("id_produto", "categoria_produto_pt")
)

df_itens_slim = (
    df_itens
    .select("id_pedido", "id_produto", "id_vendedor")
    .distinct()
)

df_base = (
    df_avaliacoes
    .select("id_avaliacao", "id_pedido", "nota_avaliacao")
    .join(df_itens_slim, on="id_pedido", how="inner")
    .join(df_produtos_cat, on="id_produto", how="left")
    .join(
        df_vendedores.select(
            "id_vendedor",
            F.col("cidade").alias("cidade_vendedor"),
            F.col("estado").alias("estado_vendedor"),
        ),
        on="id_vendedor", how="left"
    )
    .withColumn(
        "categoria_produto_pt",
        F.coalesce(F.col("categoria_produto_pt"), F.lit("Sem Categoria"))
    )
    .withColumn(
        "estado_vendedor",
        F.coalesce(F.col("estado_vendedor"), F.lit("Não Informado"))
    )
)

df_gold = (
    df_base
    .groupBy("categoria_produto_pt", "id_vendedor", "estado_vendedor")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.count(
            F.when(F.col("nota_avaliacao") >= 4, True)
        ).alias("total_avaliacoes_positivas"),
        F.count(
            F.when(F.col("nota_avaliacao") <= 2, True)
        ).alias("total_avaliacoes_negativas"),
    )
    .withColumn(
        "percentual_satisfacao",
        F.round(
            F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes") * 100, 2
        )
    )
    .withColumnRenamed("categoria_produto_pt", "categoria_produto")
    .withColumnRenamed("id_vendedor", "nome_vendedor")
    .orderBy(F.col("avaliacao_media").desc(), F.col("total_avaliacoes").desc())
    .withColumn("dt_referencia", F.to_date(F.current_timestamp()))
    .withColumn("_gold_load_at", F.current_timestamp())
)

total = df_gold.count()
nulls_cat  = df_gold.filter(F.col("categoria_produto").isNull()).count()
nulls_med  = df_gold.filter(F.col("avaliacao_media").isNull()).count()
nulls_sat  = df_gold.filter(F.col("percentual_satisfacao").isNull()).count()

assert nulls_cat == 0, "Nulos em categoria_produto."
assert nulls_med == 0, "Nulos em avaliacao_media."
assert nulls_sat == 0, "Nulos em percentual_satisfacao."

TABLE_NAME = "ecommerce_lakehouse.gold.fat_avaliacoes_clientes"

df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"{TABLE_NAME} salva com {total} registros.")

ecommerce_lakehouse.gold.fat_avaliacoes_clientes salva com 6491 registros.


Visualização qualitativa de:
1. O Produto MAIS bem avaliado.
2. O Produto MENOS bem avaliado.
3. O Vendedor MAIS bem avaliado.
4. O Vendedor MENOS bem avaliado.

In [0]:
df_rank_produto = (
    df_base
    .groupBy("id_produto", "categoria_produto_pt")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    )
    # Volume mínimo para evitar produtos com 1 avaliação nota 5 no topo
    .filter(F.col("total_avaliacoes") >= 10)
)

df_rank_vendedor = (
    df_base
    .groupBy("id_vendedor", "estado_vendedor")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    )
    .filter(F.col("total_avaliacoes") >= 10)
)

df_melhor_produto = (
    df_rank_produto
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
    .select(
        F.col("id_produto").alias("produto"),
        F.col("categoria_produto_pt").alias("categoria"),
        F.col("avaliacao_media"),
        F.col("total_avaliacoes"),
    )
)

print("\nProduto MAIS Bem Avaliado")
display(df_melhor_produto)

df_pior_produto = (
    df_rank_produto
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
    .select(
        F.col("id_produto").alias("produto"),
        F.col("categoria_produto_pt").alias("categoria"),
        F.col("avaliacao_media"),
        F.col("total_avaliacoes"),
    )
)

print("\nProduto MENOS Bem Avaliado")
display(df_pior_produto)

df_melhor_vendedor = (
    df_rank_vendedor
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
    .select(
        F.col("id_vendedor").alias("vendedor"),
        F.col("estado_vendedor"),
        F.col("avaliacao_media"),
        F.col("total_avaliacoes"),
    )
)

print("\nVendedor MAIS Bem Avaliado")
display(df_melhor_vendedor)

df_pior_vendedor = (
    df_rank_vendedor
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
    .select(
        F.col("id_vendedor").alias("vendedor"),
        F.col("estado_vendedor"),
        F.col("avaliacao_media"),
        F.col("total_avaliacoes"),
    )
)

print("\nVendedor MENOS Bem Avaliado")
display(df_pior_vendedor)


Produto MAIS Bem Avaliado ⭐


produto,categoria,avaliacao_media,total_avaliacoes
2722b7e5f68e776d18fe901638034e54,beleza_saude,5.0,12



Produto MENOS Bem Avaliado 💔


produto,categoria,avaliacao_media,total_avaliacoes
fd0065af7f09af4b82a0ca8f3eed1852,automotivo,1.18,11



Vendedor MAIS Bem Avaliado 🏅


vendedor,estado_vendedor,avaliacao_media,total_avaliacoes
48efc9d94a9834137efd9ea76b065a38,PR,5.0,32



Vendedor MENOS Bem Avaliado 💀


vendedor,estado_vendedor,avaliacao_media,total_avaliacoes
4342d4b2ba6b161468c63a7e7cfce593,RJ,1.28,18
